# Notebook 0: The Retrieval Substrate

**Capture, Score, Gate, Promote, Decay.** Five steps, same names, same order, in every notebook. This notebook builds none of them. It hands you the store they all sit on, so that by the time you write the gate in notebook 2, the database is a solved problem.

By the end of this notebook you will know which tier you are on, you will have a collection with a memory-oriented schema, the Brightwater Supply Co. handbook will be seeded into it, and hybrid search will work.

If you are new to vector databases: no worries, everything in here is handed to you working. If you have built RAG before: this is the part you already know, with a few extra columns. If you run agents in production: look hard at the schema and tell me what is missing.

## Which Weaviate are you talking to?

Setup fails for somebody in every room, so there are three tiers and one variable, `WEAVIATE_MODE` in `.env`. The code above the connection is identical on all three.

| Tier | `WEAVIATE_MODE` | What it is |
|---|---|---|
| 1 | `docker` | The local container from `docker-compose.yml`. Primary. |
| 2 | `embedded` | Weaviate Embedded, running inside this Python process. No Docker. Mac and Linux. |
| 3 | `cloud` | A shared Weaviate Cloud sandbox, one collection per attendee. Windows, or nothing local works. |

To switch: edit `.env`, change the one line, restart the kernel, run this notebook again. On the cloud tier also set `ATTENDEE_HANDLE` so your collection does not collide with the person next to you.

In [1]:
# Make `import memory` work whether Jupyter opened the repo root or the notebooks folder.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "memory" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

with contextlib.redirect_stderr(io.StringIO()):  # a noisy third-party import warning we do not need in the room
    import weaviate.classes.init  # noqa: F401

from collections import Counter

from memory import connect, describe_tier, ensure_collection, fetch_all, recall
from memory.retrieve import show
from memory.seed import seed_store

print(describe_tier())

WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


In [2]:
client = connect()
print("connected:", client.is_ready(), "| Weaviate", client.get_meta().get("version"))

connected: True | Weaviate 1.32.9


If that cell raised, read the error. Each one names the next tier to try.

## A schema built for judging, not just finding

A RAG chunk carries text and a vector. That is enough to find it again. It is not enough to decide, weeks later, whether to keep trusting it.

A memory record has to carry enough to be judged: who said it, when, in what context, how sure they were, which tier it has earned, who corroborated it, who endorsed it, when a human should look at it again, and whether it is still alive. Every one of those fields is used by a later step. None of them is decoration.

`ensure_collection(reset=True)` drops and recreates your collection. That is the right call in a workshop. Run it whenever you want a clean start.

In [3]:
collection = ensure_collection(client, reset=True)

from memory.schema import PROPERTIES

step_for = {
    "text": "what was observed", "topic": "what was observed",
    "source": "Capture", "source_type": "Capture", "context": "Capture",
    "confidence": "Capture", "captured_at": "Capture", "candidate_id": "Capture",
    "tier": "Promote", "scope": "Promote", "corroboration_count": "Promote",
    "corroborated_by": "Promote", "endorsed_by": "Promote", "promoted_at": "Promote",
    "expires_at": "Decay", "review_at": "Decay", "last_used": "Decay",
    "use_count": "Decay", "status": "Decay",
}
for p in PROPERTIES:
    print(f"{p.name:20s} {str(p.dataType).replace('DataType.', ''):12s} {step_for[p.name]}")

text                 TEXT         what was observed
topic                TEXT         what was observed
source               TEXT         Capture
source_type          TEXT         Capture
context              TEXT         Capture
confidence           NUMBER       Capture
captured_at          DATE         Capture
candidate_id         TEXT         Capture
tier                 TEXT         Promote
scope                TEXT         Promote
corroboration_count  INT          Promote
corroborated_by      TEXT_ARRAY   Promote
endorsed_by          TEXT         Promote
promoted_at          DATE         Promote
expires_at           DATE         Decay
review_at            DATE         Decay
last_used            DATE         Decay
use_count            INT          Decay
status               TEXT         Decay


Two things to notice. Vectors are supplied by the client, not by a Weaviate module, so the embedding choice stays open and nothing here depends on an API key. And the text fields used for filtering (`tier`, `scope`, `status`, `source`) are tokenized as whole fields, so a filter on `tier = org` means exactly that.

## Seed the handbook

Brightwater Supply Co. is a fictional warehouse and office supply company in Portland. Fifty short facts about its policies, people, systems, products, and customers. Fictional people, fictional systems, fictional account numbers.

They land directly in the **org** tier, endorsed by the handbook owner. That is the one path that skips the gate: a document loader with a named human owner. Everything else, from here on, is a candidate and goes through Capture, Score, and Gate.

The vectors come from `data/vector_cache.json`, so this takes seconds and does not need the embedding model.

In [4]:
n = seed_store(collection)
records = fetch_all(collection)
print(f"seeded {n}, store holds {len(records)}")
print("by tier: ", dict(Counter(r["tier"] for r in records)))
print("by topic:", dict(Counter(r["topic"] for r in records)))

seeded 50, store holds 50
by tier:  {'org': 50}
by topic: {'general': 1, 'product': 4, 'policy': 17, 'customer': 5, 'people': 8, 'systems': 7, 'process': 8}


In [5]:
# One record, every field. This is what a governed memory looks like at rest.
example = next(r for r in records if r["candidate_id"] == "seed-001")
for k, v in example.items():
    print(f"{k:20s} {v}")

uuid                 c17caad4-0baa-434d-bb30-7a02aa7c8e92
use_count            0
captured_at          2026-09-16 00:08:05.421427+00:00
endorsed_by          user:priya.natarajan
corroborated_by      ['doc:support-playbook']
corroboration_count  1
tier                 org
expires_at           2999-01-01 00:00:00+00:00
review_at            2027-03-15 00:08:05.421427+00:00
context              handbook ingest
promoted_at          2026-09-16 00:08:05.421427+00:00
scope                org
topic                policy
text                 Refunds over $500 require approval from a support manager before they are issued.
last_used            2026-09-16 00:08:05.421427+00:00
candidate_id         seed-001
status               active
source_type          document
source               doc:support-playbook
confidence           0.95


Read the dates. The handbook was captured 30 days ago, it never expires on its own (`expires_at` is a stand-in for never), and a human is asked to look at it again in 180 days. Those numbers are the Decay rules for the org tier, and notebook 3 will move the clock to show them firing.

## Read it back: hybrid search

`recall()` is the read path, kept small on purpose because this workshop is about writes. It runs a hybrid query, dense vector plus BM25 keywords, over **active** records only, and it returns org records plus the one scope you name. No scope named means you see only shared org memories. That filter is the first piece of governance in the repo and it lives on the read side.

In [6]:
for q in [
    "Who approves refunds over $500?",
    "How do remote employees connect to internal systems?",
    "What is the expense receipt rule?",
]:
    print(f"\n> {q}")
    show(recall(collection, q, limit=3))


> Who approves refunds over $500?
0.862  [org    ] Aisha Karim is Head of Customer Support and approves refunds over $500.
        from doc:org-chart (document), confidence 0.95, x1
0.808  [org    ] Refunds over $500 require approval from a support manager before they are issued
        from doc:support-playbook (document), confidence 0.95, x1
0.364  [org    ] Refunds are available within 30 days of delivery for unopened items.
        from doc:support-playbook (document), confidence 0.95, x1

> How do remote employees connect to internal systems?
1.000  [org    ] Remote employees connect to internal systems through Brightwater Secure Access, 
        from doc:it-runbook (document), confidence 0.95, x1
0.362  [org    ] Keyring is the single sign-on and secrets vault for all internal systems.
        from doc:it-runbook (document), confidence 0.95, x1
0.275  [org    ] Priya Natarajan leads People Operations and owns the employee handbook.
        from doc:org-chart (document), confiden

`alpha` sets the blend. 1 is pure vector, 0 is pure keyword. Try the same query at both ends. Keyword search finds the records that literally say VPN. Vector search also pulls in the password rule, which is about the same part of the business even though it shares no words with the query. Later, the contradiction check in Score leans on this: a candidate that disagrees with a stored fact usually sits very close to it in vector space.

In [7]:
for alpha in (0.0, 1.0):
    print(f"\n> VPN access problems   (alpha={alpha})")
    show(recall(collection, "VPN access problems", limit=3, alpha=alpha))


> VPN access problems   (alpha=0.0)
1.000  [org    ] Remote employees connect to internal systems through Brightwater Secure Access, 
        from doc:it-runbook (document), confidence 0.95, x1
0.000  [org    ] Jordan Blake leads IT and is the escalation contact for VPN and single sign-on p
        from doc:org-chart (document), confidence 0.95, x1

> VPN access problems   (alpha=1.0)
1.000  [org    ] Remote employees connect to internal systems through Brightwater Secure Access, 
        from doc:it-runbook (document), confidence 0.95, x1
0.943  [org    ] Jordan Blake leads IT and is the escalation contact for VPN and single sign-on p
        from doc:org-chart (document), confidence 0.95, x1
0.486  [org    ] Passwords on company systems rotate every 90 days and must be at least 14 charac
        from doc:it-runbook (document), confidence 0.95, x1


## Embeddings: local, cached

The model is `all-MiniLM-L6-v2`, 384 dimensions, about 90 MB, loaded with sentence-transformers. No account, no key. Every text that ships with the workshop is already in the cache, so the model is only needed for sentences you type yourself. This cell checks whether it loads on your machine. If it does not, you can still do every notebook with the shipped examples.

In [8]:
from memory.embed import cosine, embed_one, model_available

if model_available():
    v = embed_one("Who signs off on big discounts?")
    near = embed_one("Discounts above 15% on any order need sign-off from the Vice President of Sales, Lena Marsh.")
    far = embed_one("Warehouse scanners sync inventory to Ledgerstone every 15 minutes.")
    print(f"{len(v)} dimensions")
    print(f"similarity to the discount rule:  {cosine(v, near):.2f}")
    print(f"similarity to the scanner fact:   {cosine(v, far):.2f}")
else:
    print("The embedding model did not load. Stick to the shipped example texts; they are all cached.")

W1015 19:08:07.569000 8839 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384 dimensions
similarity to the discount rule:  0.64
similarity to the scanner fact:   0.16


## Fell behind, or something broke?

This one cell does everything above: connect, rebuild the collection, seed it. Run it any time you want a known-good store before moving on.

In [9]:
if "client" in dir():
    client.close()
client = connect()
collection = ensure_collection(client, reset=True)
print("reseeded", seed_store(collection), "records on", describe_tier())

reseeded 50 records on WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord


## What was Weaviate-specific here, and what was not

Weaviate-specific: all of `memory/connect.py` (about forty lines, and the only module you would rewrite for another store), the `Property` and `DataType` declarations in `memory/schema.py`, the hybrid query and its `alpha` knob, and the batch insert plus the index wait in `memory/seed.py`.

Portable: the field list itself, the rule that the handbook is the only ungated write path, and the idea that reads are filtered by tier and scope. Qdrant, Milvus, and pgvector all hold this schema and all filter on it. Hybrid search exists on each of them under a different name.

## Green check

In [10]:
count = len(fetch_all(collection))
assert count == 50, f"expected 50 seed records, found {count}"
print(f"\u2705 {describe_tier()}")
print(f"\u2705 {count} handbook records in the org tier. Ready for notebook 1.")
client.close()

✅ WEAVIATE_MODE=docker (tier 1, local Docker container), collection MemoryRecord
✅ 50 handbook records in the org tier. Ready for notebook 1.
